In [2]:
import analysis
from CNN_NAS.ChildCNNModel import ChildCNNModel
# Some magic so that the notebook will reload the external python script file any time you edit and save the .py file;
%load_ext autoreload
%autoreload 2

In [2]:
import torch
import torch.nn as nn
import time
from torch.utils.data import DataLoader
import os

import utils

import logging
logging.basicConfig(level=logging.INFO, filename=os.path.join(os.getcwd(), 'log.log'), filemode='w')

logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

device = utils.get_device_available()
print(torch.__version__)
print(device)

2.4.1
cuda


In [4]:
def is_valid_encoding(encoding):
    if len(encoding) < 3 or encoding[0] != "START" or encoding[-1] != "END":
        return False

    for i in range(1, len(encoding) - 1):
        if not encoding[i].isnumeric():
            return False

    return True


def split_dataset(data, labels, split_ratio=0.8):
    dataset = torch.utils.data.TensorDataset(data, labels)
    train_size = int(split_ratio * len(dataset))
    test_size = len(dataset) - train_size

    train_set, test_set = torch.utils.data.random_split(dataset, [train_size, test_size])

    train_data, train_labels = zip(*train_set)
    train_data = torch.stack(train_data)
    train_labels = torch.stack(train_labels)

    test_data, test_labels = zip(*test_set)
    test_data = torch.stack(test_data)
    test_labels = torch.stack(test_labels)

    return (train_data, train_labels), (test_data, test_labels)

## CIFAR-100

In [4]:
dataset="cifar"

data_path = utils.check_cifar_dataset_exists()

dataset_train_data,dataset_train_label = (torch.load(data_path + f'{dataset}/train_data.pt', weights_only=True), torch.load(data_path + f'{dataset}/train_label.pt', weights_only=True))

dataset_test_data,dataset_test_label = (torch.load(data_path + f'{dataset}/test_data.pt', weights_only=True), torch.load(data_path + f'{dataset}/test_label.pt', weights_only=True))


num_channels = 1

if len(dataset_train_data.size())==4:
    num_channels=dataset_train_data.size(1)



num_classes = dataset_train_label.unique().size(0)
height = dataset_train_data.size(-2)
width = dataset_train_data.size(-1)

print(f"Height: {height}")
print(f"Width: {width}")
print(f"Number of channels: {num_channels}")
print(f"Number of classes:  {num_classes}")



Height: 32
Width: 32
Number of channels: 3
Number of classes:  10


## Load  predefined model encoding

In [5]:
import predefined_models
# Defined by data

base_model_encoding_dict = {}

base_model_encoding_dict["Benchmark_Model"] = predefined_models.get_benchmarkModel(input_channels=num_channels, output_dim=num_classes)
base_model_encoding_dict["Lenet"] = predefined_models.get_lenet(input_channels=num_channels, output_dim=num_classes)
base_model_encoding_dict["VGG_11"] = predefined_models.get_vgg11(input_channels=num_channels, output_dim=num_classes)
# base_model_encoding_dict["Alexnet"] = predefined_models.get_alexnet(input_channels=num_channels, output_dim=num_classes)
# 

# base_model = predefined_models.get_benchmarkModel(input_channels=num_channels, output_dim=num_classes)

# print(base_model)

In [7]:
for base_model in base_model_encoding_dict:
    print(base_model)
    print(ChildCNNModel(base_model_encoding_dict[base_model], num_channels,height,width, num_classes))
    

Benchmark_Model
ChildCNNModel(
  (model): Sequential(
    (0): Conv2d(3, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(128, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (9): Flatten(start_dim=1, end_dim=-1)
    (10): Linear(in_features=512, out_features=512, bias=True)
    (11): ReLU()
    (12): Linear(in_features=512, out_features=256, bias=True)
    (13): ReLU()
    (14): Linear(in_features=256, out_features=10, bias=True)
  )
)
Lenet
ChildCNNModel(
  (model): Sequential(
    (0): Conv2d(3, 50, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool

In [7]:
from CNN_NAS.ChildCNNModel import ChildCNNModel

#Load and run
# logger.info("###############################################")
# logger.info("STARTING TRAINING")
# logger.info("###############################################")


# print(model)
utils.cleanup_child_model()
print("Done cleaning")
for base_model in base_model_encoding_dict:
    total_epochs=10
    name = base_model
    for i in range(3):
        model = ChildCNNModel(base_model_encoding_dict[base_model], num_channels,height,width, num_classes).to(device)
        loss,train_time = model.train_model(data=dataset_train_data,label=dataset_train_label,epochs=total_epochs,dataset_name=dataset)
        test_accuracy = model.evaluate_model(data=dataset_test_data,labels=dataset_test_label,dataset_name=dataset)
        print(f"Model {name} Train loss:{loss} Test Accuracy: {test_accuracy} with total epochs {total_epochs} in dataset {dataset} for time {train_time}")
        total_epochs += 10
        utils.cleanup_child_model(model)

Done cleaning


KeyboardInterrupt: 

## CIFAR

In [9]:
def benchmark_dataset(dataset_name):
    dataset=dataset_name
    
    if dataset== "cifar100":
        data_path = utils.check_cifar_dataset_exists()
    elif dataset== "cifar":
        data_path = utils.check_cifar_dataset_exists()
    elif dataset== "mnist":
        data_path = utils.check_mnist_dataset_exists()
    
    dataset_train_data,dataset_train_label = (torch.load(data_path + f'{dataset}/train_data.pt', weights_only=True), torch.load(data_path + f'{dataset}/train_label.pt', weights_only=True))
    
    dataset_test_data,dataset_test_label = (torch.load(data_path + f'{dataset}/test_data.pt', weights_only=True), torch.load(data_path + f'{dataset}/test_label.pt', weights_only=True))
    
    
    num_channels = 1
    
    if len(dataset_train_data.size())==4:
        num_channels=dataset_train_data.size(1)
    
    
    num_classes = dataset_train_label.unique().size(0)
    height = dataset_train_data.size(-2)
    width = dataset_train_data.size(-1)
    
    print(f"Height: {height}")
    print(f"Width: {width}")
    print(f"Number of channels: {num_channels}")
    print(f"Number of classes:  {num_classes}")
    
    import predefined_models
    base_model_encoding_dict = {}
    
    base_model_encoding_dict["Benchmark_Model"] = predefined_models.get_benchmarkModel(input_channels=num_channels, output_dim=num_classes)
    base_model_encoding_dict["Lenet"] = predefined_models.get_lenet(input_channels=num_channels, output_dim=num_classes)
    base_model_encoding_dict["VGG_11"] = predefined_models.get_vgg11(input_channels=num_channels, output_dim=num_classes)
    
    from CNN_NAS.ChildCNNModel import ChildCNNModel

    #Load and run
    # logger.info("###############################################")
    # logger.info("STARTING TRAINING")
    # logger.info("###############################################")
    
    
    # print(model)
    utils.cleanup_child_model()
    print("Done cleaning")
    for base_model in base_model_encoding_dict:
        total_epochs=10
        name = base_model
        for i in range(3):
            model = ChildCNNModel(base_model_encoding_dict[base_model], num_channels,height,width, num_classes).to(device)
            loss,train_time = model.train_model(data=dataset_train_data,label=dataset_train_label,epochs=total_epochs,dataset_name=dataset)
            test_accuracy = model.evaluate_model(data=dataset_test_data,labels=dataset_test_label,dataset_name=dataset)
            print(f"Model {name} Train loss:{loss} Test Accuracy: {test_accuracy} with total epochs {total_epochs} in dataset {dataset} for time {train_time}")
            total_epochs += 10
            utils.cleanup_child_model(model)



In [10]:
benchmark_dataset("cifar")

Height: 32
Width: 32
Number of channels: 3
Number of classes:  10
Done cleaning
Model Benchmark_Model Train loss:0.6706520057916642 Test Accuracy: (0.7738, 0.6541137820482255) with total epochs 10 in dataset cifar for time 41.14300203323364
Model Benchmark_Model Train loss:0.5190912814140319 Test Accuracy: (0.8095, 0.5748951834440231) with total epochs 20 in dataset cifar for time 90.95293641090393
Model Benchmark_Model Train loss:0.4400870418548584 Test Accuracy: (0.8167, 0.5625523740053177) with total epochs 30 in dataset cifar for time 124.61483263969421
Model Lenet Train loss:0.7650219378471375 Test Accuracy: (0.7545, 0.7144281446933747) with total epochs 10 in dataset cifar for time 35.61004567146301
Model Lenet Train loss:0.6383145699501037 Test Accuracy: (0.7765, 0.6483760488033294) with total epochs 20 in dataset cifar for time 76.47411680221558
Model Lenet Train loss:0.5180736609697342 Test Accuracy: (0.8185, 0.5460353457927704) with total epochs 30 in dataset cifar for time 1

In [19]:
def benchmark_dataset_model(dataset_name,experiment_name,model_encoding):
    dataset=dataset_name
    
    if dataset== "cifar100":
        data_path = utils.check_cifar_dataset_exists()
    elif dataset== "cifar":
        data_path = utils.check_cifar_dataset_exists()
    elif dataset== "mnist":
        data_path = utils.check_mnist_dataset_exists()
    
    dataset_train_data,dataset_train_label = (torch.load(data_path + f'{dataset}/train_data.pt', weights_only=True), torch.load(data_path + f'{dataset}/train_label.pt', weights_only=True))
    
    dataset_test_data,dataset_test_label = (torch.load(data_path + f'{dataset}/test_data.pt', weights_only=True), torch.load(data_path + f'{dataset}/test_label.pt', weights_only=True))
    
    
    
    
    num_channels = 1
    
    if len(dataset_train_data.size())==4:
        num_channels=dataset_train_data.size(1)
    
    
    num_classes = dataset_train_label.unique().size(0)
    height = dataset_train_data.size(-2)
    width = dataset_train_data.size(-1)
    
    print(f"Height: {height}")
    print(f"Width: {width}")
    print(f"Number of channels: {num_channels}")
    print(f"Number of classes:  {num_classes}")
    
    from CNN_NAS.ChildCNNModel import ChildCNNModel

    #Load and run
    logger.info("###############################################")
    logger.info("STARTING TRAINING")
    logger.info("###############################################")
    
    utils.cleanup_child_model()
    print("Done cleaning")
    total_epochs=10
    for i in range(3):
        model = ChildCNNModel(encoding=model_encoding, input_channels=num_channels,height=height,width=width, output_dim=num_classes).to(device)
        loss,train_time = model.train_model(data=dataset_train_data,label=dataset_train_label,epochs=total_epochs,dataset_name=dataset)
        test_accuracy = model.evaluate_model(data=dataset_test_data,labels=dataset_test_label,dataset_name=dataset)
        print(f"Model {experiment_name} Train loss:{loss} Test Accuracy: {test_accuracy} with total epochs {total_epochs} in dataset {dataset} for time {train_time}")
        total_epochs += 10
        utils.cleanup_child_model(model)



In [22]:
import predefined_models
import utils
import pandas as pd

def get_combined_df(experiment_name):
    df_train_log = pd.read_csv(os.path.join("Results", "Controller", "CNN", f"{experiment_name}.csv"))
    df_stats = pd.read_csv(os.path.join("LogFiles", "Output", f"{experiment_name}_log.csv"))

    return  pd.concat((df_train_log, df_stats), axis=1)

def find_best_row_in_first_300(df,limit=300):
    df = df.head(limit)
    return combined_df.loc[df['max_accuracy'].idxmax()]

dir_path = os.path.join("Results", "Controller", "CNN")


for filename in os.listdir(dir_path):
    if "CIFAR_100" in filename.upper():
        dataset_name = "cifar100"
        output_dim=100
    else:
        dataset_name = "cifar"
        output_dim=10
        
    base_model = predefined_models.get_benchmarkModel(input_channels=3,output_dim=output_dim)
        
    combined_df = get_combined_df(filename[:-4])
    best_row = find_best_row_in_first_300(combined_df)
    print(filename)
    print(best_row)
    
    conv_encoding = []

    for i in range(1, 4):  
        layer = (
            str(int(best_row[f'layer_{i}_channels'])),
            str(int(best_row[f'layer_{i}_filter'])),
            str(int(best_row[f'layer_{i}_padding']))
        )
        conv_encoding.append(layer)
    
    
    model_encoding  = utils.replace_multiple_conv_layers(base_model=base_model, generated_layers=conv_encoding, positions=[0,1,2], input_channels=3)
    print(model_encoding)
    
    if filename=="Exp_2_3_Layer_CIFAR_10.csv":
        benchmark_dataset_model(dataset_name=dataset_name,experiment_name=filename[:-4],model_encoding=model_encoding)
        



EXP4_CIFAR_100_Fresh.csv
train_loss                  1.710893
val_loss                    1.909748
test_acc                    0.496900
train_time                 61.406869
policy_gradient             0.190126
max_accuracy                0.496900
baseline                    0.448329
iteration                 285.000000
layer_1_channels          128.000000
layer_1_filter              3.000000
layer_1_padding             1.000000
layer_1_param_count      3584.000000
layer_2_channels          256.000000
layer_2_filter              3.000000
layer_2_padding             1.000000
layer_2_param_count    295168.000000
layer_3_channels          256.000000
layer_3_filter              3.000000
layer_3_padding             2.000000
layer_3_param_count    590080.000000
total_param_count      888832.000000
Name: 285, dtype: float64
[[[3, 128, 3, 1], [2, 2], [128, 256, 3, 1], [2, 2], [256, 256, 3, 2], [2, 2]], [[1024, 512], [512, 256], [256, 100]]]
EXP4_CIFAR_100_Pre_Trained.csv
train_loss             

KeyboardInterrupt: 

In [25]:
base_model = predefined_models.get_benchmarkModel(input_channels=3,output_dim=10)
layers = [('256', '3', '0'), ('256', '5', '1')]
model_encoding  = utils.replace_multiple_conv_layers(base_model=base_model, generated_layers=layers, positions=[1,2], input_channels=3)
model = ChildCNNModel(encoding=model_encoding, input_channels=3,height=32,width=32, output_dim=10).to(device)

model

ChildCNNModel(
  (model): Sequential(
    (0): Conv2d(3, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(128, 128, kernel_size=(1, 1), stride=(1, 1), padding=(2, 2))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(128, 64, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (9): Flatten(start_dim=1, end_dim=-1)
    (10): Linear(in_features=1600, out_features=512, bias=True)
    (11): ReLU()
    (12): Linear(in_features=512, out_features=256, bias=True)
    (13): ReLU()
    (14): Linear(in_features=256, out_features=10, bias=True)
  )
)

In [26]:
dataset="cifar"

data_path = utils.check_cifar_dataset_exists()

dataset_train_data,dataset_train_label = (torch.load(data_path + f'{dataset}/train_data.pt', weights_only=True), torch.load(data_path + f'{dataset}/train_label.pt', weights_only=True))

dataset_test_data,dataset_test_label = (torch.load(data_path + f'{dataset}/test_data.pt', weights_only=True), torch.load(data_path + f'{dataset}/test_label.pt', weights_only=True))


num_channels = 1

if len(dataset_train_data.size())==4:
    num_channels=dataset_train_data.size(1)



num_classes = dataset_train_label.unique().size(0)
height = dataset_train_data.size(-2)
width = dataset_train_data.size(-1)

print(f"Height: {height}")
print(f"Width: {width}")
print(f"Number of channels: {num_channels}")
print(f"Number of classes:  {num_classes}")


Height: 32
Width: 32
Number of channels: 3
Number of classes:  10


In [27]:
for i in range(3):
    model.train_model(data=dataset_train_data,label=dataset_train_label,epochs=10,dataset_name=dataset)
    print(model.evaluate_model(data=dataset_test_data,labels=dataset_test_label,dataset_name=dataset))

(0.7875, 0.6169862723350525)


KeyboardInterrupt: 

In [29]:
import  analysis

df = analysis.get_full_training_log_data("EXP_1_CIFAR10_META")

✅ Fixed padding captured and exported 459 models to LogFiles/Output/EXP_1_CIFAR10_META_log.csv


In [31]:
df[df["test_acc"] == 0.8164]

,train_loss,val_loss,test_acc,train_time,policy_gradient,max_accuracy,baseline,iteration,layer_1_channels,layer_1_filter,...,layer_1_param_count,layer_2_channels,layer_2_filter,layer_2_padding,layer_2_param_count,layer_3_channels,layer_3_filter,layer_3_padding,layer_3_param_count,total_param_count
240,0.530159,0.548929,0.8164,106.963175,0.276213,0.8164,0.779766,240.0,128.0,3.0,...,3584.0,256.0,3.0,0.0,295168.0,256.0,5.0,1.0,1638656.0,1937408.0


In [14]:
from CNN_NAS.ChildCNNModel import ChildCNNModel 

def count_learnable_parameters(model):
    return sum(p.numel() for p in model.model.parameters() if p.requires_grad)

base_model_encoding_dict["Benchmark_Model"] = predefined_models.get_benchmarkModel(input_channels=num_channels, output_dim=num_classes)
base_model_encoding_dict["Lenet"] = predefined_models.get_lenet(input_channels=num_channels, output_dim=num_classes)
base_model_encoding_dict["VGG_11"] = predefined_models.get_vgg11(input_channels=num_channels, output_dim=num_classes)

for base_model in base_model_encoding_dict:
    model = ChildCNNModel(base_model_encoding_dict[base_model], num_channels,height,width, num_classes).to(device)
    print(model.model)
    print(f"{base_model} : {count_learnable_parameters(model)}")


Sequential(
  (0): Conv2d(3, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (1): ReLU()
  (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (3): Conv2d(128, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (4): ReLU()
  (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (6): Conv2d(64, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (7): ReLU()
  (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (9): Flatten(start_dim=1, end_dim=-1)
  (10): Linear(in_features=512, out_features=512, bias=True)
  (11): ReLU()
  (12): Linear(in_features=512, out_features=256, bias=True)
  (13): ReLU()
  (14): Linear(in_features=256, out_features=10, bias=True)
)
Benchmark_Model : 492394
Sequential(
  (0): Conv2d(3, 50, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (1): ReLU()
  (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (3): Conv2d(50, 100